# AfroViewz 05: Baselines

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/JBWandera/Capstone/blob/afroviewz/notebooks/05_baselines.ipynb)

**Workflow stage:** modelling. **Runtime:** T4 GPU (Runtime, Change runtime type).

A new model only means something next to a baseline. Two are set here, on the new leak free test split:

* **E0, the v1 capstone model** (RF-DETR Base, trained on the old dataset only), evaluated the correct way. This is the number to beat.
* **E1, YOLO26 small**, the most common CNN detector, trained on the new dataset. YOLO26 is AGPL 3.0 licensed, so it is a comparison only and never ships in AfroViewz.

Both write predictions for the valid and test splits to `runs/<id>/`; notebook 07 scores every run the same way.

## Setup

In [ ]:
# Works on Colab and locally. On Colab: mounts Drive, gets the code, installs it.
import os, sys, subprocess
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    if not os.path.exists("/content/Capstone"):
        subprocess.run(["git", "clone", "-q", "-b", "afroviewz", "https://github.com/JBWandera/Capstone.git", "/content/Capstone"], check=True)
    else:
        subprocess.run(["git", "-C", "/content/Capstone", "pull", "-q"], check=True)
    os.chdir("/content/Capstone")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", ".[train]"], check=True)
    sys.path.insert(0, "/content/Capstone/src")
from afroviewz import paths
print("data root:", paths.data_root())


In [ ]:
import torch
assert torch.cuda.is_available(), "Runtime > Change runtime type > T4 GPU"
print(torch.cuda.get_device_name(0))


In [ ]:
# Copy the dataset from Drive to the fast local disk once per session.
import zipfile, shutil
from pathlib import Path
DATASET = "afroviewz-v1"
zip_on_drive = paths.data_root() / "processed" / f"{DATASET}.zip"
LOCAL = Path("/content/datasets") if IN_COLAB else paths.processed_dir(DATASET).parent
DATASET_DIR = LOCAL / DATASET / "rfdetr"
if not (DATASET_DIR / "train" / "_annotations.coco.json").exists():
    LOCAL.mkdir(parents=True, exist_ok=True)
    shutil.copy(zip_on_drive, LOCAL / zip_on_drive.name)
    with zipfile.ZipFile(LOCAL / zip_on_drive.name) as z:
        z.extractall(LOCAL)
for s in ("train", "valid", "test"):
    n = len(list((DATASET_DIR / s).glob("*.jpg")))
    print(f"{s}: {n} images")


## E0: the v1 capstone model

The checkpoint is the one saved in April in `MyDrive/Capstone/model_outputs/weights/`. Predictions are kept down to a score of 0.001 so mAP is measured on the full precision and recall curve.

In [ ]:
from rfdetr import RFDETRBase
from afroviewz.train import runner

V1_CKPT = Path("/content/drive/MyDrive/Capstone/model_outputs/weights/checkpoint_best_total.pth") if IN_COLAB \
    else Path(os.environ.get("V1_CKPT", "checkpoint_best_total.pth"))
e0_dir = runner.runs_root() / "E0_v1_capstone"
e0_dir.mkdir(parents=True, exist_ok=True)
v1 = RFDETRBase(pretrain_weights=str(V1_CKPT))
v1.optimize_for_inference()
for split in ("valid", "test"):
    preds, ms = runner.predict_split(runner.rfdetr_predict_fn(v1), DATASET_DIR / split)
    runner.save_predictions(e0_dir, split, preds, ms)
    print(split, len(preds), "predictions,", round(ms, 1), "ms per image")


## E1: YOLO26 small baseline

In [ ]:
!pip install -q ultralytics
e1_dir = runner.train_yolo("E1_yolo26s", DATASET_DIR)


In [ ]:
from ultralytics import YOLO
yolo = YOLO(str(e1_dir / "train" / "weights" / "best.pt"))
for split in ("valid", "test"):
    preds, ms = runner.predict_split(runner.yolo_predict_fn(yolo), DATASET_DIR / split)
    runner.save_predictions(e1_dir, split, preds, ms)
    print(split, len(preds), "predictions,", round(ms, 1), "ms per image")


**Next:** notebook 06 trains the RF-DETR sizes; notebook 07 compares everything.